In [17]:
%pip install xgboost


   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.5/69.5 MB 485.9 kB/s eta 0:02:22
   ---------------------------------------- 0.5/69.5 MB 485.9 kB/s eta 0:02:22
   ---------------------------------------- 0.8/69.5 MB 501.8 kB/s eta 0:02:17
   ---------------------------------------- 0.8/69.5 MB 501.8 kB/s eta 0:02:17
   ---------------------------------------- 0.8/69.5 MB 501.8 kB/s eta 0:02:17
    --------------------------------------- 1.0/69.5 MB 518.0 kB/s eta 0:02:13
    --------------------------------------- 1.0/69.5 MB 518.0 kB/s eta 0:02:13
    -----------------------


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.tree import DecisionTreeRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from xgboost import XGBRegressor


In [3]:
df=pd.read_csv(r'data\StudentsPerformance.csv')

In [4]:
df


,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75
...,...,...,...,...,...,...,...,...
995,female,group E,master's degree,standard,completed,88,99,95
996,male,group C,high school,free/reduced,none,62,55,55
997,female,group C,high school,free/reduced,completed,59,71,65
998,female,group D,some college,standard,completed,68,78,77


In [10]:
x=df.drop(columns=['math score'])
y=df['math score']

In [13]:
num_features=x.select_dtypes(exclude='object').columns
cat_features=x.select_dtypes(include='object').columns
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
num_trans=StandardScaler()
oh_trans=OneHotEncoder()
prepro=ColumnTransformer([
    ("OneHotEncoder",oh_trans,cat_features),
    ("StandardScaler",num_trans,num_features),

])

C:\Users\RAJDEEP\AppData\Local\Temp\ipykernel_16028\2292322722.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features=x.select_dtypes(include='object').columns


In [14]:
x=prepro.fit_transform(x)

In [18]:
x_t,x_test,y_t,y_test=train_test_split(x,y,test_size=0.3)
x_t.shape

(700, 19)

In [41]:
def evaluate(true,predicted):
    mae=mean_absolute_error(true,predicted)
    # mse=np.sqrt(mean_squared_error(true,predicted))
    r2=r2_score(true,predicted)
    rmse=np.sqrt(mean_squared_error(true,predicted))
    return mae,rmse,r2

In [44]:
models={
    'Linear reg':LinearRegression(),
    'Lasso':Lasso(),
    'Ridge':Ridge(),
    'K neighbor':KNeighborsRegressor(),
    'Decision tree':DecisionTreeRegressor(),
    'random foresst':RandomForestRegressor(),
    'catboost':CatBoostRegressor(),
    'AdaBoost':AdaBoostRegressor(),
    'Xgboost':XGBRegressor()
}
model_list=[]
r2_list=[]
for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(x_t,y_t)
    y_t_pred=model.predict(x_t)
    y_test_pred=model.predict(x_test)
    model_train_mae,model_train_rmse,model_train_r2=evaluate(y_t,y_t_pred)
    model_test_mae,model_test_rmse,model_test_r2=evaluate(y_test,y_test_pred)
    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])
    print(f"Train RMSE: {model_train_rmse:.4f}")
    print(f"Train MAE : {model_train_mae:.4f}")
    print(f"Train R2  : {model_train_r2:.4f}")

    print(f"Test RMSE : {model_test_rmse:.4f}")
    print(f"Test MAE  : {model_test_mae:.4f}")
    print(f"Test R2   : {model_test_r2:.4f}")


Linear reg
Train RMSE: 5.2726
Train MAE : 4.2180
Train R2  : 0.8825
Test RMSE : 5.5138
Test MAE  : 4.3543
Test R2   : 0.8577
Lasso
Train RMSE: 6.4816
Train MAE : 5.0852
Train R2  : 0.8224
Test RMSE : 6.4930
Test MAE  : 5.2092
Test R2   : 0.8026
Ridge
Train RMSE: 5.2728
Train MAE : 4.2168
Train R2  : 0.8825
Test RMSE : 5.5189
Test MAE  : 4.3565
Test R2   : 0.8574
K neighbor
Train RMSE: 5.5302
Train MAE : 4.3934
Train R2  : 0.8707
Test RMSE : 7.6291
Test MAE  : 6.1373
Test R2   : 0.7275
Decision tree
Train RMSE: 0.1336
Train MAE : 0.0071
Train R2  : 0.9999
Test RMSE : 8.2745
Test MAE  : 6.6333
Test R2   : 0.6795
random foresst
Train RMSE: 2.3283
Train MAE : 1.8297
Train R2  : 0.9771
Test RMSE : 6.1004
Test MAE  : 4.8578
Test R2   : 0.8258
Learning rate set to 0.038699
0:	learn: 14.9583571	total: 946us	remaining: 946ms
1:	learn: 14.5758335	total: 1.77ms	remaining: 884ms
2:	learn: 14.2298387	total: 2.51ms	remaining: 834ms
3:	learn: 13.8671158	total: 3.52ms	remaining: 876ms
4:	learn: 13.529